# Analysis — the short version

One bug fix and four added analyses, run against the released results. Two
minutes, no GPU.

The full pipeline stays in `PreProcessing_patched.ipynb`; this notebook starts
from its output (`raw_results.csv`).

**Needs:** `raw_results.csv`, `table1_dataset_characteristics.csv`,
`moderator_data.csv`

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, levene

# resolve the data directory whether the notebook is run from notebooks/,
# from the repository root, or from a directory holding the CSVs directly
def _find_dir(marker, *cands):
    for c in cands:
        if (Path(c) / marker).exists():
            return Path(c).resolve()
    raise FileNotFoundError(f"{marker} not found in {cands}")

DATA   = _find_dir("raw_results.csv", ".", "../data", "data", "../../data")
TABLES = _find_dir("table4_pivot.csv", ".", "../tables", "tables", str(DATA))
print(f"data   : {DATA}")
print(f"tables : {TABLES}")

LEVELS = ["P1","P2","P3","P4","P5","P6","P7","P8a","P8b"]

raw  = pd.read_csv(DATA/"raw_results.csv")
meta = pd.read_csv(DATA/"table1_dataset_characteristics.csv").set_index("dataset")
ok   = raw[raw.fold_mean.notna()].copy()
ok["task"] = ok.dataset.map(meta.task)
ok["IR"]   = ok.dataset.map(meta.imbalance_ratio)

def W(piv):
    """Kendall's W = rank-based effect size."""
    piv = piv.dropna(); n, k = piv.shape
    r = piv.rank(axis=1, ascending=False).values
    return 12*n/(k*(k+1)) * np.sum((r.mean(0)-(k+1)/2)**2) / (n*(k-1))

prep = ok.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")[LEVELS]
algo = (ok[ok.algo_id != "A10_TabPFN"]
          .groupby(["dataset","algo_id"])["fold_mean"].mean().unstack("algo_id"))

print(f"{len(ok):,} valid cells of {len(raw):,} | {ok.dataset.nunique()} datasets contributing")

## 1. The headline (RQ1)

In [ ]:
wp, wa = W(prep), W(algo)
print(f"preprocessing effect  W = {wp:.4f}")
print(f"algorithm effect      W = {wa:.4f}")
print(f"ratio                     {wa/wp:.2f}x   -> learner choice dominates")
print(f"contrast              D = {wp-wa:+.4f}")

## 2. Where the preprocessing effect lives

All of it is at one rung: the introduction of feature scaling.

In [ ]:
for lo, hi in zip(LEVELS[:-1], LEVELS[1:]):
    c = prep[[lo, hi]].dropna()
    _, p = wilcoxon(c[hi], c[lo], alternative="greater")
    mark = "  <-- the only step that matters" if p < .001 and (c[hi]-c[lo]).mean() > .01 else ""
    print(f"  {lo:>3} -> {hi:<3}  mean diff {(c[hi]-c[lo]).mean():+.4f}   p={p:.4f}{mark}")

## 3. For whom it matters

Sensitivity varies roughly sixfold across learners. For SVM and $k$-NN the
preprocessing effect exceeds the pooled algorithm effect, so the ordering above
reverses once you fix the learner.

In [ ]:
rows = []
for a, g in ok.groupby("algo_id"):
    p = g.groupby("pipeline")["fold_mean"].mean()
    rows.append((a, p.get("P1"), p.get("P4"), p["P8b"] - p["P1"]))
sens = (pd.DataFrame(rows, columns=["algorithm","P1","P4","terminal_gain"])
          .sort_values("terminal_gain", ascending=False).round(4))
print(sens.to_string(index=False))

groups = []
for a, g in ok.groupby("algo_id"):
    c = g.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")[["P1","P8b"]].dropna()
    if len(c) >= 2: groups.append((c.P8b - c.P1).values)
F, p = levene(*groups)
print(f"\nLevene across {len(groups)} families: F={F:.3f}, p={p:.2g} -> sensitivity is not uniform")

## 4. Added after review: the portfolio matters

Removing the weakest learner cuts the headline ratio substantially. The sign is
robust; the magnitude is not.

In [ ]:
for name, drop in [("all 10", []), ("- decision tree", ["A2_DecTree"]),
                   ("- dec. tree, kNN", ["A2_DecTree","A11_kNN"])]:
    s  = ok[~ok.algo_id.isin(list(drop) + ["A10_TabPFN"])]
    pp = s.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")[LEVELS]
    pa = s.groupby(["dataset","algo_id"])["fold_mean"].mean().unstack("algo_id")
    ds = sorted(set(pp.dropna().index) & set(pa.dropna().index))
    wp_, wa_ = W(pp.loc[ds]), W(pa.loc[ds])
    print(f"  {name:18s} k={pa.shape[1]:2d}  ratio {wa_/wp_:.2f}x   D={wp_-wa_:+.4f}")

## 5. Added after review: the metric hides one rung

$P_{8b}$ reweights classes, but ROC-AUC is threshold-free and cannot see it.

In [ ]:
cls = ok[ok.task == "classification"]
for label, sub in [("IR>=5 (weighting on)", cls[cls.IR >= 5]), ("all", cls)]:
    for name, col in [("ROC-AUC", "fold_mean"), ("bal. accuracy", "bacc_mean")]:
        c = (sub.groupby(["dataset","pipeline"])[col].mean()
                .unstack("pipeline")[["P8a","P8b"]].dropna())
        _, p = wilcoxon(c.P8b, c.P8a, alternative="greater")
        print(f"  {label:22s} {name:14s} diff {(c.P8b-c.P8a).mean():+.4f}  p={p:.4f}")

## 6. Added after review: the three empty datasets

Both causes are properties of the harness, not the data, and both are
recoverable.

In [ ]:
for d in [d for d, g in raw.groupby("dataset") if g.fold_mean.notna().sum() == 0]:
    m = meta.loc[d]
    if m.task == "classification":
        cause = "rarest class below the 10 CV folds -> StratifiedKFold raises in the outer loop"
    else:
        cause = "single numeric column -> transformer could not emit a 2-D matrix"
    print(f"  {d:16s} classes={m.n_classes:.0f} IR={m.imbalance_ratio:.0f} "
          f"num/cat={m.n_numeric:.0f}/{m.n_categorical:.0f}\n{'':18s}{cause}")

## 7. The bug that was fixed

`mord.LogisticAT` needs labels in a contiguous range 0..C−1. Minimal-sufficiency
labels have gaps ($P_5$ is never minimal), so the fit raised and the original
`except` clause quietly returned a baseline instead.

In [ ]:
from mord import LogisticAT
y = np.array([0,0,1,1,3,5,6,8] * 15)          # gapped, like the real labels
X = np.random.RandomState(0).randn(len(y), 3)

try:
    LogisticAT(alpha=1.0).fit(X, y)
except Exception as e:
    print(f"before: {type(e).__name__}: {e}")

cls_ = np.unique(y); enc = {c: i for i, c in enumerate(cls_)}
m = LogisticAT(alpha=1.0).fit(X, np.array([enc[v] for v in y]))
pred = cls_[np.clip(m.predict(X).astype(int), 0, len(cls_)-1)]
print(f"after : fits; predictions decode to {sorted(set(pred))}; "
      f"{len(m.predict_proba(X))} confidences")

---

The fix restored RQ4: the ordinal model turns out to be *refuted* (0.6490 against
a 0.7092 base rate), while nearest-dataset retrieval *beats* the baseline
(0.7799). The full leave-one-dataset-out evaluation is in
`pipeline_source_updated.ipynb`, together with 273 assertions checking every number in
the paper.